In [ ]:
import numpy as np  
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns 

from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
import xgboost as xgb
from sklearn.compose import TransformedTargetRegressor


# Data Loade

In [ ]:
from zipfile import ZipFile

In [ ]:
zip_path = '/Users/mikkelpedersen/Desktop/project_vs_studio/random projekter/insurance.zip'

with ZipFile(zip_path, 'r') as insurance:
    file_list = insurance.namelist()
    print("Files in zip:", file_list)
    
    # Use the first CSV (adjust logic if multiple files exist)
    target_file = file_list[0]

    with insurance.open(target_file) as insurance_df:
        df = pd.read_csv(insurance_df)

# Data inspection

In [ ]:
df.head(10)

In [ ]:
class DataPlotter():
    def __init__(self, Dataframe : pd.DataFrame):
        self.df = Dataframe
        
    def histogram(self, column):
        if column not in self.df:
            raise ValueError('The column does not exist in the dataframe')
        plt.figure(figsize=(10,6))
        sns.countplot(x=self.df[column])
        plt.title(f'Historgram of {column}')
        plt.show()
        print(self.df[column].value_counts())
        
        
    def scatter(self, x_column, y_column):
        if x_column not in self.df or y_column not in self.df:
            raise ValueError('The columns does not exist in the df')
        plt.figure(figsize=(10,6))
        sns.scatterplot(x=self.df[x_column], y=self.df[y_column])
        plt.title(f'Scatterplot of x: {x_column} and y: {y_column}')
        plt.xlabel(x_column)
        plt.ylabel(y_column)
        plt.show()
        
    def class_pairplot(self):
        plt.figure(figsize=(10,10))
        sns.pairplot(self.df)
        plt.show()
        display(self.df.describe())
        
        
    def box_plot(self, x_col, y_col):
        plt.figure(figsize=(10,5))
        sns.boxplot(x=self.df[x_col], y=self.df[y_col], medianprops={"color": "r", "linewidth": 2}, notch=True)
        plt.title(f'Boxplot of x: {x_col}, and y: {y_col}')
        plt.show()
        
        
    def info(self, val_count_col):
        display(self.df)
        print('-'*50)
        for col in self.df.select_dtypes(include=['float64', 'int64']):
            print(f' the mean of {col}: {round(self.df[col].mean(),2)}') 
        print('-'*50)
        print('The amount of missing values')
        print(self.df.isnull().sum())
        print('-'*50)
        print('Describe')
        display(self.df.describe())
        print('-'*50)
        print(f'Value counts for {col}')
        print(self.df[val_count_col].value_counts())

In [ ]:
plotter = DataPlotter(df)

In [ ]:
df.info()

In [ ]:
for i in df.select_dtypes(include=['object']):
    plotter.box_plot('charges', i)

In [ ]:
plotter.info('children')

In [ ]:
plotter.histogram('sex')

In [ ]:
plotter.scatter('bmi', 'charges')

In [ ]:
plotter.class_pairplot()

# Data Preprocessing

In [ ]:
df_copy = df.copy(deep=True)
target_cop = df['charges']

In [ ]:
print(df.select_dtypes(include='object')), print(df.select_dtypes(include=['int64', 'float64']))

In [ ]:
class Preprocessing ():
    def __init__(self, DataFrame : pd.DataFrame):
        self.df = DataFrame
        self.scaler = StandardScaler()
        self.encoder = OneHotEncoder(sparse_output=False)
        
    def numeric_prepors(self):
        num_cols = self.df.select_dtypes(include=['float64', 'int64']).columns
        num_cols = [col for col in num_cols if col != 'charges']
        self.df[num_cols] = self.scaler.fit_transform(self.df[num_cols])
        print('Numeric columns sclaed')
        display(self.df[num_cols].head())         
        
        
    def categorical_prepors(self):
        cat_col = self.df.select_dtypes(include='object').columns
        # Fit and transform the categorical columns
        encoded = self.encoder.fit_transform(self.df[cat_col])
        encoded_df = pd.DataFrame(
            encoded,
            columns=self.encoder.get_feature_names_out(cat_col),
            index=self.df.index
        )

        # Drop original categorical columns and concatenate encoded ones
        self.df.drop(columns=cat_col, inplace=True)
        self.df = pd.concat([self.df, encoded_df], axis=1)
        print('Categorical encoded columns:')
        display(self.df.head(10))   
    
    def get_df(self):
        return self.df

In [ ]:
preprocessor = Preprocessing(df)

In [ ]:
preprocessor.numeric_prepors()
preprocessor.categorical_prepors()
df = preprocessor.get_df()

In [ ]:
df.head(10)

# Models

In [ ]:
X = df.drop(columns='charges')
y = df['charges']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [ ]:
class Models:
    def __init__(self, X_train, y_train, X_test, y_test):
        self.X_train = X_train
        self.y_train = y_train.to_numpy().reshape(-1, 1)
        self.X_test = X_test
        self.y_test = y_test.to_numpy().reshape(-1, 1)
        self.target_scaler = StandardScaler()
        
        # Base models
        self.models = {
            'LinearRegression': LinearRegression(),
            'Ridge': Ridge(),
            'DecisionTree': DecisionTreeRegressor(random_state=42),
            'RandomForest': RandomForestRegressor(random_state=42),
            'GradientBoosting': GradientBoostingRegressor(random_state=42)
        }

        # Corresponding parameter grids
        self.param_grids = {
            'DecisionTree': {
                'max_depth': [None, 5, 10, 20],
                'min_samples_split': [2, 5, 10],
                'min_samples_leaf': [1, 2, 4],
                'criterion': ['squared_error', 'friedman_mse', 'absolute_error']
            },
            'RandomForest': {
                'n_estimators': [100, 200],
                'max_depth': [None, 10, 20],
                'min_samples_split': [2, 5],
                'min_samples_leaf': [1, 2],
                'bootstrap': [True, False]
            },
            'GradientBoosting': {
                'n_estimators': [100, 200],
                'learning_rate': [0.01, 0.1],
                'max_depth': [3, 5],
                'min_samples_split': [2, 5],
                'min_samples_leaf': [1, 2],
                'subsample': [0.8, 1.0]
            }
        }

        self.trained_models = {}
        self.results = {}

    def model_grid_search(self):
        y_scaled = self.target_scaler.fit_transform(self.y_train)
        for name, model in self.models.items():
            if name in self.param_grids:
                print(f"Performing grid search for {name}")
                grid_search = GridSearchCV(
                    model,
                    self.param_grids[name],
                    cv=10,
                    scoring='neg_mean_absolute_error',
                    n_jobs=-1
                )
                grid_search.fit(self.X_train, y_scaled.ravel())  # flatten target for sklearn
                best_model = grid_search.best_estimator_
                self.models[name] = best_model
                print(f"Best params for {name}: {grid_search.best_params_}")
            else:
                print(f"No grid search for {name}, using default parameters")
        
        print("Grid search complete.\n")

    def train_all(self):
        y_scaled = self.target_scaler.transform(self.y_train)  # Don't refit here
        for name, model in self.models.items():
            model.fit(self.X_train, y_scaled.ravel())
            self.trained_models[name] = model
            y_pred_scaled = model.predict(self.X_test).reshape(-1, 1)
            print(y_pred_scaled)
            y_pred = self.target_scaler.inverse_transform(y_pred_scaled)
        
            mae = mean_absolute_error(self.y_test, y_pred)
            mape = mean_absolute_percentage_error(self.y_test, y_pred)
            r2 = r2_score(self.y_test, y_pred)
            
            self.results[name] = {
                'mae': mae,
                'mape': mape,
                'r2': r2
            }

        print("Model training and evaluation complete.\n")


    def model_evaluation(self):
        print('Model evaluation result')
        for name, metric in self.results.items():
            print(f"Model: {name}, MAE: {metric['mae']:.3f}, MAPE: {metric['mape']:.3f}, R2 Score: {metric['r2']:.3f}")

    def plot_predictions(self, model_name):
        if model_name not in self.trained_models:
            raise ValueError(f"{model_name} is not trained or does not exist.")
        model = self.trained_models[model_name]
        y_pred_scaled = model.predict(self.X_test).reshape(-1, 1)
        y_pred = self.target_scaler.inverse_transform(y_pred_scaled)
        plt.figure(figsize=(8, 6))
        sns.scatterplot(x=self.y_test.ravel(), y=y_pred.ravel())
        plt.xlabel("Actual")
        plt.ylabel("Predicted")
        plt.title(f"{model_name} Predictions")
        plt.plot([self.y_test.min(), self.y_test.max()], [self.y_test.min(), self.y_test.max()], 'r--')
        plt.show()


In [ ]:
model_manager = Models(X_train, y_train, X_test, y_test)

In [ ]:
model_manager.model_grid_search()

In [ ]:
model_manager.train_all()
model_manager.model_evaluation()

In [ ]:
df['charges'].mean()

In [ ]:
model_manager.plot_predictions('RandomForest')